# Extraccion datos API gestion de Fugas Von Roll

Interaccion con la API mediante request.

VonRoll usa el metodo POST, enviando y recibiendo JSON nativos; aunque puede usar GET para consultas de mapas.

- Cabecera direciones API: https://{project}/
El {proyecto} en la cabecera de direcciones puede ser la instalación en local, o bien la de la nube de VonRoll, que sería https://api.infraport.world/ifp/v1

De esta forma, por ejemplo para la autenticacion usariamos https://api.infraport.world/ifp/v1/api/sf/auth/login


In [ ]:
# Cargo librerias

import requests
import json
import xml.etree.ElementTree as ET

# Tablas
import pandas as pd

# Codificaciones
import base64

#Acceso al SO
import sys
import os
from datetime import datetime, timedelta
import time

# Gestion de zonas horarias
from zoneinfo import ZoneInfo

# Cargar las variables del archivo .env
from dotenv import load_dotenv
load_dotenv()

## Consulta de dispositivos asociados a una cuenta con su posicion

Devuelve tabla con los dispositivos de localización de fugas asociados a la cuenta, con su identificador, el de su posición y su número de serie, además de las coordenadas de la posición. 

Si un dispositivo se cambia de posición, sus coordenadas cambiarán en esta tabla. No se hace a menudo, pero conviene guardar registro.

Si los dispositivos se han asociado pero aún no han sido desplegados en campo, sus coordenadas aparecen en cero.

No informa sobre su estado actual: sirve para cualquier tipo de dispositivo Vonroll.

In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # Extraigo el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")


    #############################
    # Consulta de localizadores asociados a una cuenta
    
    # Configurar las cabeceras (Headers) para las próximas consultas de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }
    
    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")
    
    # Endpoint: GET base + /api/sf/customer/{customer}/loggers/list/compact
    devices_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/loggers/list/compact'
      
    devices_response = requests.get(devices_url, headers=headers)
    
    if devices_response.status_code == 200:
        loggers_list = devices_response.json()
        
        # Convertimos la respuesta JSON directamente a un DataFrame de Pandas
        df_loggers = pd.DataFrame(loggers_list)
        
        # Guardar tabla de inventario
        if not df_loggers.empty:
            df_loggers.to_csv("dispositivos_vonroll.csv", index=False, encoding="utf-8-sig")
            print(f"Se han procesado {len(df_loggers)} aparatos y guardado en 'dispositivos_vonroll.csv'.")
        else:
            print("No se encontraron registros individuales de loggers.")
 
    else:
        print(f"\nLogin correcto, pero error al leer dispositivos ({devices_response.status_code}).")
        print(f"Respuesta del servidor: {devices_response.text}")
        
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)

## Lista de aparatos actualmente en fuga

Lista simple de identificaciones de aparatos que estan marcando fuga.


In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # 3. Extraer el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")

    ###############################
    # 4. Configurar las cabeceras (Headers) para las próximas consultas de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }

    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")
    
    # Endpoint: GET base + /api/sf/customer/{customer}/loggers/list/compact
    devices_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/maps/loggers'

    # Filtro aparatos con fuga y que hayan medido (elimino no instalados)
    filtro_objeto = {
        "has_leakage": [True],
        "no_measurement": []
    }
    
    # Parámetros de la consulta (Cambiamos light a False para intentar traer datos completos)
    params = {
        "filter": json.dumps(filtro_objeto),
        "light": "true",
        "history": "2" # ultimo estado conocido solamente
    }


    devices_response = requests.get(devices_url, headers=headers, params=params)
    
    if devices_response.status_code == 200:
        loggers_leak = devices_response.json()
        
        # EXTRAER LA LISTA INTERNA: Extraemos 'articles' que es donde residen los loggers
        lista_fugas_hoy = loggers_leak.get("articles", [])
        
        if lista_fugas_hoy:
            # Convertimos la lista de diccionarios directamente a un DataFrame
            df_leak_now = pd.DataFrame(lista_fugas_hoy)
            
            # Guardar tabla de inventario
            if not df_leak_now.empty:
                df_leak_now.to_csv("lista_loggers_fuga_hoy.csv", index=False, encoding="utf-8-sig")
                print(f"Se han encontrado {len(df_leak_now)} aparatos en fuga y guardado en 'lista_loggers_fuga_hoy.csv'.")
            else:
                print("No se encontraron loggers en posiciones con fugas.")
 
            print(f"\nSe han encontrado {len(df_leak_now)} dispositivos en fuga")
            
        else:
            print("\nLa API respondió correctamente, pero la lista de fugas está vacía.")

    else:
        print(f"\nLogin correcto, pero error al leer dispositivos ({devices_response.status_code}).")
        print(f"Respuesta del servidor: {devices_response.text}")
        
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)

## Consulta de valores detallados para cada logger por su identificador 

Esta interface permite obtener datos detallados de un logger, por ejemplo cada uno de los que tenían fuga en la lista anterior.

No permite consultas masivas. Hay que hacer un bucle con una consulta para cada logger, con un delay entre lanzamientos ¿2-5 sg?

Devuelve nivel de bateria y última transmisión, además de otros parámetros que se hubiesen definido, como alarmas, etc. 

Se podría lanzar para todos, cambiando la lista por la general de dispositivos, y comprobar baterias o problemas de comunicacion. 

Sirve para cualquier tipo de dispositivo, como caudalimetros, etc. 

In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # 3. Extraer el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")

    ###############################
    # 4. Configurar las cabeceras (Headers) para las próximas consultas de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }

    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")
    
    devices_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/loggers'

    # Recupero la lista de la respuesta anterior (solo los que tienen fuga)
    if lista_fugas_hoy:
        # Creo el DataFrame base con los IDs y UUIDs que sí funcionan
        df_base = pd.DataFrame(lista_fugas_hoy)
        print(f"\n¡Éxito! Identificados {len(df_base)} dispositivos en el mapa.")
        print("Extrayendo la telemetría detallada de cada aparato...")
            
        # Creo una lista vacia para meter los detalles que vaya sacando
        detalles_completos = []
            
        # Iteramos por cada dispositivo usando su ID único
        for index, fila in df_base.iterrows():
            logger_id = fila['id']
                
            # Construimos la URL para el detalle individual (siguiendo el estándar de la API)
            detalle_url = f'{devices_url}/{logger_id}'
                
            # Hacemos la petición limpia, sin parámetros ni filtros complejos
            res_detalle = requests.get(detalle_url, headers=headers)
                
            if res_detalle.status_code == 200:
                datos_logger = res_detalle.json()
                detalles_completos.append(datos_logger)
            else:
                print(f"No se pudieron obtener datos del logger {logger_id} (Código {res_detalle.status_code})")
            
        # Si logramos obtener detalles, creamos el DataFrame definitivo
        if detalles_completos:
            df_fugas = pd.DataFrame(detalles_completos)
            
            print(f"\n-> Proceso completado. Se han encontrado {len(df_fugas)} aparatos con fuga activa:")
            
            # Exportar el DataFrame a un archivo CSV
            df_fugas.to_csv('detalles_completos_loggers_fuga_actual.csv', index=False, encoding='utf-8-sig')
            print("¡Archivo 'detalles_completos_loggers_fuga_actual.csv' guardado con éxito!")
        
        else:
            print("No se pudo extraer el detalle de ningún dispositivo.")
                
    else:
        print("\nLa API respondió correctamente, pero la lista de artículos está vacía.")
    
       
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)


# Extraccion de fugas activas 

Ademas de los aparatos que están escuchando las fugas, podemos extraer los tramos con una fuga detectada, correlada entre los loggers, en el registro de los operarios buscafugas utilizan para comprobar si es o no fuga.

In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    
    # Exatrigo el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")

    ###############################
    # Configuro cabeceras (Headers) para las próximas consultas de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }
    
    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")
    
    leaks_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/leaks'

    params_leaks = {
        "itemsPerPage": 100,      # Aumentamos el límite para capturar el histórico completo (improbable mas de 100 correlaciones dia)
        "order[number]": "desc"
    }

    print("Descargando el listado operativo de fugas...")
    leaks_response = requests.get(leaks_url, headers=headers, params=params_leaks)

    if leaks_response.status_code == 200:
        leaks_raw = leaks_response.json()
    
        if leaks_raw:
            # Aplanamos el JSON para desempaquetar el diccionario interno 'location'
            df_leaks = pd.json_normalize(leaks_raw)
        
   
            print(f"¡Éxito! Se han importado {len(df_leaks)} registros de fugas.")
        
            # Filtro rápido en Pandas: separamos confirmadas de las que están pendientes (to_check)
            df_pendientes = df_leaks[df_leaks['status'] == 'to_check']
            print(f"-> Hay {len(df_pendientes)} alertas pendientes de verificación en campo de {len(df_leaks)} en total.")

            # Exportar el DataFrame a un archivo CSV
            df_leaks.to_csv('detalles_fugas_estado.csv', index=False, encoding='utf-8-sig')

            print("¡Archivo 'detalles_fugas_estado.csv' guardado con éxito!")

        else:
            print("La consulta se realizó correctamente pero el histórico está vacío.")
    else:
        print(f"Error al acceder al endpoint de fugas (Código {leaks_response.status_code})")
       
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)
    

## Informes de segmentacion: tramos con fuga para una fecha: correlations/segment-report/sync

Indican las fugas que puede haber para un dia concreto, sobre los tramos. Identifica posiciones de escucha, aparatos detectores, valores de ruido en cada posición y correlación entre pares, con las coordenadas de la fuga entre ellas. Indica la probabilidad de fuga (score %).

Se debe analizar espacialmente ya que una fuga puede escucharse entre varios loggers y aparecerá tantas veces como combinaciones de pares puedan hacerse.

Se indica en el payload la fecha del dia anterior a la consulta, para asegurar que haya datos comunicados.


In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # 3. Extraer el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")


    ######################################
    
    # Configuro cabeceras para la consulta de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }

    # Calculo la fecha del dia anterior a la consulta
    hoy = datetime.now()
    ayer = hoy - timedelta(days=1)
    fecha_consulta = ayer.strftime("%Y-%m-%d")

    
    # Creo el paydload con la fecha
    payload = {
        "date": str(fecha_consulta)
    }
    
    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")
    
    segment_rpt_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/correlations/segment-report/sync'
      
    # USAR con metodo POST añadiendo el paidload con la fecha como string
    segment_rpt_response = requests.post(segment_rpt_url, headers=headers, json=payload)
    
    if segment_rpt_response.status_code == 200:
        segment_rpt_list = segment_rpt_response.json()
        
        # Convertimos la respuesta JSON directamente a un DataFrame de Pandas
        df_segment_rpt = pd.DataFrame(segment_rpt_list)
        print(f"\nSe han encontrado {len(df_segment_rpt)} tramos con fugas para esa fecha:")
        
        # Exportar el DataFrame a un archivo CSV
        df_segment_rpt.to_csv('registro_fugas_ayer.csv', index=False, encoding='utf-8-sig')
        print("¡Archivo 'registro_fugas_ayer.csv' guardado con éxito!")

    else:
        print(f"\nLogin correcto, pero error al leer los datos solicitados ({segment_rpt_response.status_code}).")
        print(f"Respuesta del servidor: {segment_rpt_response.text}")
        
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)

## Extraccion del ultimo archivo de sonido obtenido por un logger

En la consulta pasamos el ID del logger que obtuvimos en la lista de loggers asociados a la cuenta. Pero en la devolución del archivo de sonido, para su nombre incluye el serial number del aparato y la fecha de la grabacion. 

Por otra parte, ademas del archivo también devuelve el identificador interno del dispositivo (dev_id), y su posición para poder geoposicionarlo en sus coordenadas.

In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # 3. Extraer el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")

    ###############################
    
    # Configuro cabeceras (Headers) para las próximas consultas de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }

    # Creo el paydload para extraer el sonido
    payload = {
        "sound": {
            "no_fm": True
        }
    }

    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")

    # Defino id Logger
    # Sustituir por el que provea el bucle al iterar sobre la lista
    LOGGER_ID = "1302378"
    
    ### Debería recorrerse diariamente la lista de todos los logger_id para extraer sus archivos de sonido.
    # 'https://api.infraport.world/ifp/v1/api/sf/customer/{customer}/loggers/{logger_id}/sound_data'
    sound_data_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/loggers/{logger_id}/sound_data'
    
    sound_response = requests.post(sound_data_url, json=payload, headers=headers)

    # Muestro la respuesta
    print(sound_response)

    ## Proceso la respuesta controlando los errores posibles
    if sound_response.status_code == 200:
        sound_list = sound_response.json()
        
        # Extraigo el nombre del archivo original con identificador, canales y fecha
        file_name = sound_list['soundFileName']

        # Limpio la cabecera del string Base64 
        # Elimino "data:audio/x-wav;base64," para dejar solo los datos puros
        base64_data = sound_list['sound'].split(',')[1]

        # Decodifico los datos binarios que lleganen texto plano codeBase64
        audio_bytes = base64.b64decode(base64_data)

        # Guardo los datos binarios como un archivo de sonido .wav
        with open(file_name, 'wb') as audio_file:
            audio_file.write(audio_bytes)

        print(f"¡Éxito! Archivo de audio guardado correctamente como: {file_name}")
        
    else:
        print(f"\nLogin correcto, pero error al leer dispositivos ({sound_response.status_code}).")
        print(f"Respuesta del servidor: {sound_response.text}")
        
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)

# Extrae serie temporal de medidas de ruido para una posicion de escucha

Medidas de ruido registradas por el logger indicado, situado en una posición, obtenidas cada media hora, a lo largo de un periodo. 

Siempre extrae los datos de la última semana. Crear historico consolidando valores in incrementar.

Solo admite un logger cada vez: crear un bucle a partir de la lista de loogers de la cuenta, y dejar tiempo 2/5 sg entre consultas.

In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # 3. Extraer el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")

    ############################
    
    # Configuro cabeceras (Headers) para las próximas consultas de datos
    headers = {
        "Authorization": f"Bearer {mi_access_token}",
        "Content-Type": "application/json"
    }

    # Creo el paydload para extraer las mediciones de la serie temporal
    query_params = {
        "templateKey": "loggerOverview",
        "navigator": "1",
    }

    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")

    # Defino id Logger
    # Sustituir por el que provea el bucle al iterar sobre la lista
    LOGGER_ID = "1302378"
    
    # 'https://api.infraport.world/ifp/v1/api/sf/customer/{customer}/logger-chart/{logger_id}'
    chart_data_url = f'https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/logger-chart/{logger_id}'
    
    chart_response = requests.get(chart_data_url, headers=headers, params=query_params)

    ## Proceso la respuesta
    if chart_response.status_code == 200:
        chart_list = chart_response.json()

        # Extraigo las variables de la lista JSON
        logger_label = chart_list['chartOptions']['title']['text']
        series_id = chart_list['series'][0]['id']
        raw_data = chart_list['series'][0]['data']

        # Creo un DataFrame estructurado
        df_chart = pd.DataFrame(raw_data, columns=['timestamp', 'value'])

        # Convierto las marcas de tiempo Unix (ms) a fechas legibles
        df_chart['date'] = pd.to_datetime(df_chart['timestamp'], unit='ms')

        # Incorporo metadatos de contexto
        df_chart['logger_label'] = logger_label
        df_chart['series_id'] = series_id

        # Reordeno columnas para que sea limpio de leer
        df_chart = df_chart[['date', 'logger_label', 'series_id', 'value', 'timestamp']]

        # Configuro un archivo CSV de salida
        output_file = f'chart_datalogger_{logger_id}.csv'

        # Comprobar si el archivo ya existe para no duplicar la cabecera
        file_exists = os.path.isfile(output_file)

        # Guardar o añadir (mode ='a') al CSV sin añadir de nuevo las columnas (header = not file exists)
        df_chart.to_csv(output_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
      
        print(f"¡Éxito! Se han volcado {len(df_chart)} registros en el archivo: {output_file}")
        
        
    else:
        print(f"\nLogin correcto, pero error al leer dispositivos ({chart_response.status_code}).")
        print(f"Respuesta del servidor: {chart_response.text}")
        
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)

# Extraccion de resumen de ruidos para un logger

Similar al anterior, obtiene solo la medición de ruido diario representativo para un solo logger, cada dia de la ultima semana. 

Solo admite un logger cada vez: crear un bucle a partir de la lista de loogers de la cuenta, y dejar tiempo 2/5 sg entre consultas.

In [ ]:
### OBTENGO EL TOKEN PARA LA CONSULTA

# Definicion de url para la validacion, a partir de una base
base_url = "https://api.infraport.world/ifp/v1" 
login_url = f"{base_url}/api/sf/auth/login"

# Credenciales
credentials = {
    "username": os.getenv("API_USER"),
    "password": os.getenv("API_PASSWORD")
}

print("Iniciando sesión en INFRAPORT...")

# Realizamos la petición POST enviando las credenciales en formato JSON nativo
login_response = requests.post(login_url, json=credentials)

if login_response.status_code == 200:
    # 3. Extraer el Token de seguridad de la respuesta JSON
    login_data = login_response.json()

    # Extraigo el diccionario interno 'token' en un primer nivel
    info_token = login_data.get('token')
    
    # Ahora saco el token de acceso en el segundo nivel
    mi_access_token = info_token.get('access_token')
    
    print("¡Conexión web exitosa! Token obtenido correctamente.")


    #################################
    
    # 4. Configurar las cabeceras (Headers) para las próximas consultas de datos
    headers = {
    "Authorization": f"Bearer {mi_access_token}",
    "accept": "application/json, text/plain, */*",
}

    # Creo el paydload para extraer el sonido
    params = {
    "templateKey": "loggerInfobar",
}

    ### Parametrizacion de consulta por cliente y logger
    
    # Extraigo la cuenta
    customer_id = os.getenv("API_CUSTOMER")
    
    # Defino id Logger
    # Sustituir por el que provea el bucle al iterar sobre la lista
    LOGGER_ID = "1302378"
   
    ## Compongo direccion para acceder al logger (el primer ID de la lista creada en dispositivos_VonRoll.csv)
    # 'https://api.infraport.world/ifp/v1/api/sf/customer/{customer}/logger-chart/{logger_id}'
    bar_data_url = f"https://api.infraport.world/ifp/v1/api/sf/customer/{customer_id}/logger-chart/{LOGGER_ID}"
    
    bar_response = requests.get(bar_data_url, headers=headers, params=params)

    ## Proceso la respuesta
    if bar_response.status_code == 200:
        data_bar = bar_response.json()

        # Extraigo las variables de la lista JSON
        logger_label = data_bar['chartOptions']['title']['text']
        print(f"--- Datos del Logger: {logger_label} ---")

        # 1. Extraer la lista de puntos de la primera serie (Micrófono/Fugas)
        # Usamos .get() o controlamos el índice por seguridad
        if "series" in data_bar and len(data_bar["series"]) > 0:
            serie_info = data_bar["series"][0]
            raw_data = serie_info["data"]  # Lista de listas: [[ts, valor], [ts, valor]...]
            sensor_name = serie_info.get("name", "sensor_valor")

            # 2. Crear el DataFrame de Pandas
            df = pd.DataFrame(raw_data, columns=["timestamp", sensor_name])

            # 3. Convertir los milisegundos a formato de fecha legible (datetime)
            df["fecha"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)

            # 4. Reordenar y limpiar las columnas para el CSV final
            # Cambiamos UTC a la zona horaria local (Madrid/Europa) si lo requieres
            df["fecha"] = df["fecha"].dt.tz_convert("Europe/Madrid")
            df = df[["fecha", sensor_name]]

            # 5. Exportar a un archivo CSV
            # mode='a' permitiria añadir registros sin poner otra vez los nombres de las columnas
            nombre_archivo = f"bar_logger_{logger_id}.csv"
            df.to_csv(nombre_archivo, mode='a', index=False, encoding="utf-8-sig")

            print(f"✅ Archivo '{nombre_archivo}' exportado con {len(df)} registros.")
            print(df.head())  # Muestra las primeras filas en la consola
    
        else:
            print("❌ No se encontraron series de datos en la respuesta de la API.")

        
    else:
        print(f"\nLogin correcto, pero error al leer dispositivos ({bar_response.status_code}).")
        print(f"Respuesta del servidor: {bar_response.text}")
        
else:
    print(f"Error de autenticación ({login_response.status_code}):")
    print(login_response.text)